# Project 3 · Part 2: Project 3 · Part 2: City rents — exploring prices across the city · Practice

**Open Fraud Labs Academy**

1. Copy your **practice key** from the Practice tab of this lesson on the Academy and paste it into the first code cell.
2. Run the first cell, then complete each exercise and run its check.
3. Every check that passes is saved to your lesson page automatically. Solve all three to unlock the quiz.

Tip: choose **File > Save a copy in Drive** to keep your work.

In [ ]:
# Step 1: paste your practice key between the quotes, then run this cell (Shift + Enter).
# Find your key on the lesson page, in the Practice tab.
OFL_KEY = ""

import pandas as pd, requests
DATA_URL = "https://raw.githubusercontent.com/Odugbile1993/openfraudlab-tiktok/main/data/loans.csv"
COURSE, LESSON = "data-science", 38

def ofl_check(exercise, check_code):
    """Runs the check for one exercise and, if it passes, records it on your Academy lesson page."""
    try:
        exec(check_code, globals())
    except AssertionError as e:
        print("Not quite yet:", e or "compare your result with the task and try again.")
        return
    except NameError as e:
        print("Something the task asks for is missing. Run your code cell first.", e)
        return
    print("Correct! Well done.")
    if not OFL_KEY.strip():
        print("Add your practice key in the first cell and run it, then run this check again to record your progress.")
        return
    try:
        r = requests.post("https://virxrqwxvgsbcnmhdwlp.supabase.co/rest/v1/rpc/practice_report",
                          headers={"apikey": "sb_publishable_8IenXF_4F5ew8hLai8RfVA_dNHq4I72", "Content-Type": "application/json"},
                          json={"p_key": OFL_KEY.strip(), "p_course": COURSE, "p_lesson": LESSON, "p_exercise": exercise}, timeout=20)
        d = r.json()
        if r.ok:
            print(f"Saved to your lesson page: {d['solved']} of {d['total']} exercises solved.")
        else:
            print("Could not save your progress:", d.get("message", r.text))
    except Exception as e:
        print("Could not reach the Academy to save your progress:", e)

print("Ready. Work through the exercises below." if OFL_KEY.strip() else "Paste your practice key above so your progress is saved.")

## Exercise 1: Rent by property type

Store the median annual rent of each property type in type_medians, and the type with the highest median in top_type.

<details><summary>Hint</summary>

rentals.groupby("property_type")["monthly_rent_usd"].median() gives the medians; .idxmax() gives the label of the largest.

</details>

In [ ]:
import pandas as pd

rentals = pd.read_csv(DATA_URL.replace("loans.csv", "projects/city_rentals.csv"))
rentals = rentals.drop_duplicates().reset_index(drop=True)
rentals["size_sqm"] = rentals["size_sqm"].fillna(rentals.groupby("property_type")["size_sqm"].transform("median"))

type_medians = ...   # median annual rent for each property type
top_type = ...       # the property type with the highest median rent

print(type_medians)
print(top_type)

In [ ]:
# Check exercise 1 (run after your code above)
ofl_check(1, "_r = pd.read_csv(DATA_URL.replace(\"loans.csv\", \"projects/city_rentals.csv\"))\n_r = _r.drop_duplicates().reset_index(drop=True)\n_r[\"size_sqm\"] = _r[\"size_sqm\"].fillna(_r.groupby(\"property_type\")[\"size_sqm\"].transform(\"median\"))\n_m = _r.groupby(\"property_type\")[\"monthly_rent_usd\"].median()\nassert isinstance(type_medians, pd.Series), \"type_medians should be a Series: rentals.groupby(\\\"property_type\\\")[\\\"monthly_rent_usd\\\"].median()\"\nassert ((type_medians.reindex(_m.index).astype(float) - _m).abs() < 1e-6).all(), \"type_medians should hold the median annual rent of each property type\"\nassert isinstance(top_type, str) and top_type == _m.idxmax(), \"top_type should be the property type with the highest median: type_medians.idxmax()\"\n")

## Exercise 2: Skew before and after the log

Take the natural log of annual rent in log_rent, then store the skew of the rent in skew_raw and the skew of log_rent in skew_log.

<details><summary>Hint</summary>

np.log(rentals["monthly_rent_usd"]) gives the logs; .skew() works on any numeric Series.

</details>

In [ ]:
import pandas as pd
import numpy as np

rentals = pd.read_csv(DATA_URL.replace("loans.csv", "projects/city_rentals.csv"))
rentals = rentals.drop_duplicates().reset_index(drop=True)
rentals["size_sqm"] = rentals["size_sqm"].fillna(rentals.groupby("property_type")["size_sqm"].transform("median"))

log_rent = ...    # natural log of monthly_rent_usd
skew_raw = ...    # skew of monthly_rent_usd
skew_log = ...    # skew of log_rent

print(skew_raw, skew_log)

In [ ]:
# Check exercise 2 (run after your code above)
ofl_check(2, "import numpy as _np\n_r = pd.read_csv(DATA_URL.replace(\"loans.csv\", \"projects/city_rentals.csv\"))\n_r = _r.drop_duplicates().reset_index(drop=True)\n_r[\"size_sqm\"] = _r[\"size_sqm\"].fillna(_r.groupby(\"property_type\")[\"size_sqm\"].transform(\"median\"))\n_lr = _np.log(_r[\"monthly_rent_usd\"])\nassert isinstance(log_rent, pd.Series) and len(log_rent) == len(_r), \"log_rent should be np.log(rentals[\\\"monthly_rent_usd\\\"])\"\nassert ((log_rent.values - _lr.values) ** 2).max() < 1e-12, \"log_rent should be the natural log of monthly_rent_usd\"\nassert skew_raw is not Ellipsis and abs(float(skew_raw) - _r[\"monthly_rent_usd\"].skew()) < 1e-6, \"skew_raw should be rentals[\\\"monthly_rent_usd\\\"].skew()\"\nassert skew_log is not Ellipsis and abs(float(skew_log) - _lr.skew()) < 1e-6, \"skew_log should be log_rent.skew()\"\n")

## Exercise 3: The furnished premium, fairly

Build a pivot table of median rent per square metre by neighbourhood (rows) and furnished (columns), divide furnished by not furnished in each neighbourhood, and store the median of those ratios in furnished_ratio.

<details><summary>Hint</summary>

rentals.pivot_table(index="neighbourhood", columns="furnished", values="rent_per_sqm", aggfunc="median"), then pt[1] / pt[0], then .median().

</details>

In [ ]:
import pandas as pd

rentals = pd.read_csv(DATA_URL.replace("loans.csv", "projects/city_rentals.csv"))
rentals = rentals.drop_duplicates().reset_index(drop=True)
rentals["size_sqm"] = rentals["size_sqm"].fillna(rentals.groupby("property_type")["size_sqm"].transform("median"))
rentals["rent_per_sqm"] = rentals["monthly_rent_usd"] / rentals["size_sqm"]

pt = ...              # pivot table: index neighbourhood, columns furnished, median rent_per_sqm
ratio = ...           # furnished (1) divided by not furnished (0), for each neighbourhood
furnished_ratio = ... # the median of ratio

print(furnished_ratio)

In [ ]:
# Check exercise 3 (run after your code above)
ofl_check(3, "_r = pd.read_csv(DATA_URL.replace(\"loans.csv\", \"projects/city_rentals.csv\"))\n_r = _r.drop_duplicates().reset_index(drop=True)\n_r[\"size_sqm\"] = _r[\"size_sqm\"].fillna(_r.groupby(\"property_type\")[\"size_sqm\"].transform(\"median\"))\n_r[\"rent_per_sqm\"] = _r[\"monthly_rent_usd\"] / _r[\"size_sqm\"]\n_pt = _r.pivot_table(index=\"neighbourhood\", columns=\"furnished\", values=\"rent_per_sqm\", aggfunc=\"median\")\n_want = (_pt[1] / _pt[0]).median()\nassert isinstance(ratio, pd.Series) and len(ratio) == len(_pt), \"ratio should have one value per neighbourhood: pt[1] / pt[0]\"\nassert furnished_ratio is not Ellipsis and not hasattr(furnished_ratio, \"__len__\"), \"furnished_ratio should be one number: ratio.median()\"\nassert abs(float(furnished_ratio) - _want) < 0.01, \"furnished_ratio should be the median, across neighbourhoods, of furnished / not furnished median rent per sqm\"\n")

---
Done? Go back to the lesson on [openfraudlabs.com/academy](https://openfraudlabs.com/academy/lesson/?course=data-science&n=38) and take the quiz.